In [9]:
import os
import librosa
import numpy as np
from tqdm import tqdm

DATA_PATH = '../Data/speech_commands/'
SAVE_PATH = '../Data/'

TARGET_WORDS = ['on', 'off', 'stop']
UNKNOWN_WORDS = ['right', 'left', 'up', 'down']

X = [] 
y = [] 

def get_mfcc(file_path):
    try:
        # 1. Load audio file
        audio, sr = librosa.load(file_path, sr=16000)
        
        # 2. FEATURE ENGINEERING: Trim silence from beginning and end
        # top_db=25 means any sound 25dB below the peak volume is considered silence and removed
        trimmed_audio, _ = librosa.effects.trim(audio, top_db=25)
        
        # Filter out extremely short noises (e.g., clicks shorter than 0.15 seconds)
        if len(trimmed_audio) < 2400: 
            return None

        # 3. Extract MFCC on the PURE spoken word
        mfccs = librosa.feature.mfcc(y=trimmed_audio, sr=sr, n_mfcc=13, hop_length=512)
        mfccs = mfccs.T # Transpose to shape (frames, 13)

        # 4. FEATURE ENGINEERING: Dynamic Temporal Pooling
        # Dynamically split the frames into 3 equal sections, regardless of length
        parts = np.array_split(mfccs, 3)
        
        # Calculate the mean for each dynamic part
        part1 = np.mean(parts[0], axis=0) # Always the start of the word
        part2 = np.mean(parts[1], axis=0) # Always the middle
        part3 = np.mean(parts[2], axis=0) # Always the end

        # Concatenate them into a single 1D array of 39 features
        features = np.concatenate((part1, part2, part3))
        
        return features
    except Exception as e:
        return None

print("Processing Target Words (with Silence Trimming)...")
for label_idx, word in enumerate(TARGET_WORDS):
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir):
        print(f"Warning: Directory for '{word}' not found.")
        continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    for file_name in tqdm(files[:1500], desc=f"Loading {word}"): 
        file_path = os.path.join(word_dir, file_name)
        mfcc_data = get_mfcc(file_path)
        if mfcc_data is not None:
            X.append(mfcc_data)
            y.append(label_idx)

print("\nProcessing Unknown Words (with Silence Trimming)...")
unknown_label_idx = len(TARGET_WORDS) 
for word in UNKNOWN_WORDS:
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir):
        continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    for file_name in tqdm(files[:375], desc=f"Loading {word} (Unknown)"): 
        file_path = os.path.join(word_dir, file_name)
        mfcc_data = get_mfcc(file_path)
        if mfcc_data is not None:
            X.append(mfcc_data)
            y.append(unknown_label_idx)

X = np.array(X)
y = np.array(y)

np.save(os.path.join(SAVE_PATH, 'X_features.npy'), X)
np.save(os.path.join(SAVE_PATH, 'y_labels.npy'), y)

print(f"\nFeature extraction complete! Saved {len(X)} robust samples.")
print(f"NEW Features shape: {X.shape} (Dynamic Temporal Pooling)")

Processing Target Words (with Silence Trimming)...


Loading on:   0%|          | 0/1500 [00:00<?, ?it/s]

Loading stop: 100%|██████████| 1500/1500 [00:16<00:00, 93.28it/s] 



Processing Unknown Words (with Silence Trimming)...


Loading down (Unknown): 100%|██████████| 375/375 [00:04<00:00, 92.87it/s]


Feature extraction complete! Saved 5995 robust samples.
NEW Features shape: (5995, 39) (Dynamic Temporal Pooling)


In [10]:
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os

DATA_PATH = '../Data/'
OUTPUT_PATH = '../edge_mcu'

# 1. Load the FULL 39-feature dataset
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. STAGE 1: Dimensionality Reduction using LDA
print("\n--- Running LDA to reduce 39 features to 3 Super-Features ---")
# n_components is always (number of classes - 1) for LDA
lda = LinearDiscriminantAnalysis(n_components=3)
X_train_lda = lda.fit_transform(X_train, y_train)
X_test_lda = lda.transform(X_test)

print(f"New Training Data Shape after LDA: {X_train_lda.shape}")

# 3. STAGE 2: Optuna Optimization for Logistic Regression
print("\n--- Starting Optuna optimization for Logistic Regression ---")
def objective(trial):
    # Tune the regularization parameter C
    c_val = trial.suggest_float('C', 1e-4, 1e2, log=True)
    
    lr = LogisticRegression(C=c_val, max_iter=5000, random_state=42)
    lr.fit(X_train_lda, y_train)
    preds = lr.predict(X_test_lda)
    
    return accuracy_score(y_test, preds)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print(f"\nBest Optuna Parameters: {study.best_params}")
print(f"Best Accuracy during CV: {study.best_value * 100:.2f}%")

# 4. TRAIN FINAL CLASSIFIER
print("\n--- Training Final Lightweight Classifier ---")
final_lr = LogisticRegression(**study.best_params, max_iter=5000, random_state=42)
final_lr.fit(X_train_lda, y_train)

preds = final_lr.predict(X_test_lda)
unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]

print("\nFinal Classification Report:")
print(classification_report(y_test, preds, target_names=target_names))

# 5. EXPORT PURE PYTHON ARRAYS FOR EDGE INFERENCE
print("\n--- Exporting LDA and Classifier Parameters to model_data_lr.py ---")

# Extract LDA parameters
lda_xbar = lda.xbar_.tolist()         # Shape: (39,) -> The mean of each feature
lda_scalings = lda.scalings_.tolist() # Shape: (39, 3) -> The transformation matrix

# Extract Logistic Regression parameters
lr_coef = final_lr.coef_.tolist()           # Shape: (4, 3) -> Weights for the 3 super-features
lr_intercept = final_lr.intercept_.tolist() # Shape: (4,) -> Bias for each class

output_file = os.path.join(OUTPUT_PATH, 'model_data_lr.py')
with open(output_file, 'w') as f:
    f.write("# Auto-Generated LDA + Logistic Regression Export\n\n")
    f.write(f"CLASSES = {target_names}\n\n")
    
    # Write LDA arrays
    f.write("LDA_XBAR = [\n")
    f.write(f"    {[round(float(val), 6) for val in lda_xbar]}\n")
    f.write("]\n\n")
    
    f.write("LDA_SCALINGS = [\n")
    for row in lda_scalings:
        f.write(f"    {[round(float(val), 6) for val in row]},\n")
    f.write("]\n\n")
    
    # Write LR arrays
    f.write("LR_COEF = [\n")
    for class_weights in lr_coef:
        f.write(f"    {[round(float(w), 6) for w in class_weights]},\n")
    f.write("]\n\n")
    
    f.write("LR_INTERCEPT = [\n")
    f.write(f"    {[round(float(i), 6) for i in lr_intercept]}\n")
    f.write("]\n")

print(f"Pipeline successfully exported to: {output_file}")

[I 2026-07-01 17:02:59,414] A new study created in memory with name: no-name-7dbd896a-e5fa-4792-8b91-a2829a75a76f
[I 2026-07-01 17:02:59,426] Trial 0 finished with value: 0.7773144286905754 and parameters: {'C': 2.5383547588560353}. Best is trial 0 with value: 0.7773144286905754.
[I 2026-07-01 17:02:59,438] Trial 1 finished with value: 0.780650542118432 and parameters: {'C': 0.011291346063873798}. Best is trial 1 with value: 0.780650542118432.
[I 2026-07-01 17:02:59,450] Trial 2 finished with value: 0.7773144286905754 and parameters: {'C': 7.668392108257607}. Best is trial 1 with value: 0.780650542118432.
[I 2026-07-01 17:02:59,458] Trial 3 finished with value: 0.7823185988323603 and parameters: {'C': 0.00043465989292766605}. Best is trial 3 with value: 0.7823185988323603.
[I 2026-07-01 17:02:59,469] Trial 4 finished with value: 0.7781484570475397 and parameters: {'C': 0.8785472719922128}. Best is trial 3 with value: 0.7823185988323603.
[I 2026-07-01 17:02:59,481] Trial 5 finished with

Loaded Data -> X shape: (5995, 39), y shape: (5995,)

--- Running LDA to reduce 39 features to 3 Super-Features ---
New Training Data Shape after LDA: (4796, 3)

--- Starting Optuna optimization for Logistic Regression ---


[I 2026-07-01 17:02:59,605] Trial 12 finished with value: 0.7823185988323603 and parameters: {'C': 0.0017756498864779588}. Best is trial 7 with value: 0.7848206839032527.
[I 2026-07-01 17:02:59,627] Trial 13 finished with value: 0.7773144286905754 and parameters: {'C': 0.2755485290944579}. Best is trial 7 with value: 0.7848206839032527.
[I 2026-07-01 17:02:59,639] Trial 14 finished with value: 0.7839866555462885 and parameters: {'C': 0.002065074072276255}. Best is trial 7 with value: 0.7848206839032527.
[I 2026-07-01 17:02:59,653] Trial 15 finished with value: 0.7789824854045038 and parameters: {'C': 0.06181032960065418}. Best is trial 7 with value: 0.7848206839032527.
[I 2026-07-01 17:02:59,667] Trial 16 finished with value: 0.7814845704753962 and parameters: {'C': 0.016494074688797412}. Best is trial 7 with value: 0.7848206839032527.
[I 2026-07-01 17:02:59,678] Trial 17 finished with value: 0.780650542118432 and parameters: {'C': 0.000575071689988148}. Best is trial 7 with value: 0.7


Best Optuna Parameters: {'C': 0.00010003996805327174}
Best Accuracy during CV: 78.73%

--- Training Final Lightweight Classifier ---

Final Classification Report:
              precision    recall  f1-score   support

          on       0.81      0.83      0.82       299
         off       0.77      0.82      0.79       300
        stop       0.85      0.81      0.83       300
     unknown       0.72      0.69      0.71       300

    accuracy                           0.79      1199
   macro avg       0.79      0.79      0.79      1199
weighted avg       0.79      0.79      0.79      1199


--- Exporting LDA and Classifier Parameters to model_data_lr.py ---
Pipeline successfully exported to: ../edge_mcu\model_data_lr.py


In [25]:
import sys
import os

sys.path.append(os.path.abspath('../edge_mcu'))

import model_data_lr

def predict_audio_class(mfcc_features):
    """
    Optimized Inference Engine with Dynamic Dimension Detection.
    """
    # Dynamically detect dimensions from the imported model
    num_original_features = len(model_data_lr.LDA_SCALINGS)      # Should be 39
    num_super_features = len(model_data_lr.LDA_SCALINGS[0])      # Should be 3 (or 2)
    num_classes = len(model_data_lr.CLASSES)
    
    # --- STAGE 1: LDA Transformation ---
    lda_features = [0.0] * num_super_features
    for i in range(num_super_features):
        sum_val = 0.0
        for j in range(num_original_features):
            sum_val += mfcc_features[j] * model_data_lr.LDA_SCALINGS[j][i]
        lda_features[i] = sum_val
        
    # --- STAGE 2: Logistic Regression Inference ---
    class_scores = [0.0] * num_classes
    for c in range(num_classes):
        # Accessing intercept correctly based on your export script
        # If model_data_lr.LR_INTERCEPT is a 1D list, use [c]
        # If it is a 2D list (as in your export code), use [0][c]
        score = model_data_lr.LR_INTERCEPT[0][c] if isinstance(model_data_lr.LR_INTERCEPT[0], list) else model_data_lr.LR_INTERCEPT[c]
        
        for i in range(num_super_features):
            score += lda_features[i] * model_data_lr.LR_COEF[c][i]
            
        class_scores[c] = score
            
    # Find the index of the highest score (Argmax)
    return class_scores.index(max(class_scores))

# --- Example Usage on Board ---
# features = [1.2, -0.5, 3.4, ...] # The 39 MFCC features extracted from the mic
# result_idx = predict_audio_class(features)
# print("Predicted Word:", model_data.CLASSES[result_idx])

import random
for j in range(10):
    i = random.randint(0, len(y_test))
    features = X_test[i]
    result_idx = predict_audio_class(features)
    print("Predicted Word:", model_data_lr.CLASSES[result_idx], "and actuall is:", y_test[i] + 1)

Predicted Word: unknown and actuall is: 3
Predicted Word: unknown and actuall is: 4
Predicted Word: unknown and actuall is: 4
Predicted Word: unknown and actuall is: 4
Predicted Word: unknown and actuall is: 3
Predicted Word: unknown and actuall is: 2
Predicted Word: unknown and actuall is: 1
Predicted Word: unknown and actuall is: 3
Predicted Word: unknown and actuall is: 2
Predicted Word: unknown and actuall is: 2
